# Ejercicio 3: Modelo Vectorial y TF-IDF

## Objetivo de la práctica

- Comprender el modelo vectorial como base para representar documentos y consultas.
- Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`
- Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

### Paso 1: Calcular la matriz TF-IDF para el corpus `data/01_corpus_turismo_500.txt`

In [3]:
import math
import unicodedata

def normalizar_palabra(palabra):
    # Eliminar tildes y convertir a minúsculas alfanuméricas
    sin_tildes = ''.join(
        c for c in unicodedata.normalize('NFD', palabra)
        if unicodedata.category(c) != 'Mn'
    )
    return ''.join(c for c in sin_tildes.lower() if c.isalnum())

def cargar_corpus(ruta):
    corpus = []
    vocabulario = set()
    with open(ruta, 'r', encoding='utf-8') as f:
        for linea in f:
            tokens = [normalizar_palabra(p) for p in linea.strip().split() if normalizar_palabra(p)]
            if tokens:
                corpus.append(tokens)
                vocabulario.update(tokens)
    return corpus, sorted(vocabulario)

def calcular_tfidf(corpus, vocabulario):
    N = len(corpus)

    # Frecuencia de término por documento (TF)
    tf_matriz = []
    for doc in corpus:
        fila = [doc.count(termino) for termino in vocabulario]
        tf_matriz.append(fila)

    # Frecuencia inversa de documento (IDF)
    idf_vector = []
    for termino in vocabulario:
        df = sum(1 for doc in corpus if termino in doc)
        idf_vector.append(round(math.log10(N / df), 4) if df else 0.0)

    return tf_matriz, idf_vector

# --- Ejecución ---
ruta = 'data/01_corpus_turismo_500.txt'
corpus, vocabulario = cargar_corpus(ruta)
tf_matriz, idf_vector = calcular_tfidf(corpus, vocabulario)

print(f"Documentos en el corpus : {len(corpus)}")
print(f"Términos únicos         : {len(vocabulario)}")

# Mostrar matriz TF y valores IDF
print("\n=== MATRIZ TÉRMINO-DOCUMENTO (TF) con IDF ===")
encabezado = "Término".ljust(20) + " | " + " | ".join(f"D{i+1}" for i in range(len(corpus))) + " | IDF"
print(encabezado)
print("-" * len(encabezado))

for idx, termino in enumerate(vocabulario):
    frecuencias = " | ".join(str(tf_matriz[doc_i][idx]) for doc_i in range(len(corpus)))
    print(f"{termino.ljust(20)} | {frecuencias} | {idf_vector[idx]:.4f}")

Documentos en el corpus : 500
Términos únicos         : 118

=== MATRIZ TÉRMINO-DOCUMENTO (TF) con IDF ===
Término              | D1 | D2 | D3 | D4 | D5 | D6 | D7 | D8 | D9 | D10 | D11 | D12 | D13 | D14 | D15 | D16 | D17 | D18 | D19 | D20 | D21 | D22 | D23 | D24 | D25 | D26 | D27 | D28 | D29 | D30 | D31 | D32 | D33 | D34 | D35 | D36 | D37 | D38 | D39 | D40 | D41 | D42 | D43 | D44 | D45 | D46 | D47 | D48 | D49 | D50 | D51 | D52 | D53 | D54 | D55 | D56 | D57 | D58 | D59 | D60 | D61 | D62 | D63 | D64 | D65 | D66 | D67 | D68 | D69 | D70 | D71 | D72 | D73 | D74 | D75 | D76 | D77 | D78 | D79 | D80 | D81 | D82 | D83 | D84 | D85 | D86 | D87 | D88 | D89 | D90 | D91 | D92 | D93 | D94 | D95 | D96 | D97 | D98 | D99 | D100 | D101 | D102 | D103 | D104 | D105 | D106 | D107 | D108 | D109 | D110 | D111 | D112 | D113 | D114 | D115 | D116 | D117 | D118 | D119 | D120 | D121 | D122 | D123 | D124 | D125 | D126 | D127 | D128 | D129 | D130 | D131 | D132 | D133 | D134 | D135 | D136 | D137 | D138 | D139 | D140 

### Paso 2: Construir el corpus `Gutenberg 1000`

El corpus `Gutenberg 1000` es un corpus compuesto por 1000 libros de Gutenberg Project

In [4]:
!pip install -q requests
import re
import time
import requests
from pathlib import Path
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

OUTPUT_FILE = "data/gutenberg_corpus.txt"  # ← ruta local
BASE_API = "https://gutendex.com/books"
MAX_BOOKS = 1000

downloaded = 0
page = 1

session = requests.Session()
retries = Retry(total=5, backoff_factor=2, status_forcelist=[429, 500, 502, 503, 504])
session.mount("https://", HTTPAdapter(max_retries=retries))

def limpiar_texto(texto):
    texto = re.sub(r'\n{3,}', '\n\n', texto)
    return texto

print("Construyendo corpus Gutenberg 1000...\n")

Path("data").mkdir(exist_ok=True)  # Crear carpeta data si no existe

with open(OUTPUT_FILE, "w", encoding="utf-8") as archivo_corpus:
    while downloaded < MAX_BOOKS:
        try:
            respuesta = session.get(f"{BASE_API}?page={page}", timeout=60)
            respuesta.raise_for_status()
        except Exception as e:
            print(f"Error cargando página {page}: {e}")
            break

        libros = respuesta.json().get("results", [])
        if not libros:
            break

        for libro in libros:
            if downloaded >= MAX_BOOKS:
                break

            titulo  = libro["title"]
            book_id = libro["id"]
            txt_url = None

            for key, value in libro["formats"].items():
                if "text/plain" in key and value:
                    txt_url = value
                    break

            if not txt_url:
                continue

            try:
                print(f"[{downloaded+1}/{MAX_BOOKS}] {titulo}")
                r = session.get(txt_url, timeout=60)
                if r.status_code != 200:
                    continue

                texto = limpiar_texto(r.text)
                archivo_corpus.write(f"\n{'='*80}\n")
                archivo_corpus.write(f"TITLE: {titulo}\nGUTENBERG ID: {book_id}\n")
                archivo_corpus.write(f"{'='*80}\n\n{texto}\n\n")

                downloaded += 1
                time.sleep(0.5)

            except Exception as e:
                print(f"Error descargando {titulo}: {e}")

        page += 1

print(f"\nListo! {downloaded} libros guardados en: {OUTPUT_FILE}")

Construyendo corpus Gutenberg 1000...

[1/1000] The City of God, Volume I


[2/1000] Frankenstein; or, the modern prometheus
[3/1000] Moby Dick; Or, The Whale
[4/1000] Pride and Prejudice
[5/1000] Crime and Punishment
[6/1000] Romeo and Juliet
[7/1000] Alice's Adventures in Wonderland
[8/1000] The origin and development of the moral ideas
[9/1000] A Room with a View
[10/1000] The strange case of Dr. Jekyll and Mr. Hyde
[11/1000] The Complete Works of William Shakespeare
[12/1000] The Count of Monte Cristo
[13/1000] Middlemarch
[14/1000] The Great Gatsby
[15/1000] The King in Yellow
[16/1000] Jane Eyre: An Autobiography
[17/1000] The Blue Castle: a novel
[18/1000] Report of the President's Commission on the Assassination of President John F. Kennedy
[19/1000] Little Women; Or, Meg, Jo, Beth, and Amy
[20/1000] The Importance of Being Earnest: A Trivial Comedy for Serious People
[21/1000] Dracula
[22/1000] Wuthering Heights
[23/1000] Adventures of Huckleberry Finn
[24/1000] The Picture of Dorian Gray
[25/1000] Twenty years after
[26/1000] The Adventures of Sherlo

[831/1000] Around the World in Eighty Days
[832/1000] Encyclopaedia Britannica, 11th Edition, "Bible" to "Bisectrix": Volume 3, Slice 7
[833/1000] Hans Holbein the Younger, Volume 1 (of 2)
[834/1000] Celtic Scotland : $b A history of ancient Alban. Volume 3 (of 3), Land and people
[835/1000] Audubon the Naturalist: A History of His Life and Time. Vol. 2 (of 2)
[836/1000] The Foundation of the Ottoman Empire; a history of the Osmanlis up to the death of Bayezid I (1300-1403)
[837/1000] The Passionate Friends
[838/1000] The Works of the Emperor Julian, Vol. 1
[839/1000] The art of money getting : $b or, golden rules for making money
[840/1000] Dante and the early astronomers
[841/1000] Lone Star Planet
[842/1000] The History and Antiquities of the Doric Race, Vol. 1 of 2
[843/1000] Cornish Characters and Strange Events
[844/1000] Aristotle
[845/1000] Criminality and economic conditions
[846/1000] The Lost Continent
[847/1000] Leyte: The Return to the Philippines
[848/1000] The Elementary

### Paso 3: Calcular la matriz TF-IDF para el corpus `Gutenberg 1000`

In [1]:
import math
import re
import unicodedata
from collections import Counter

ARCHIVO = "data/gutenberg_corpus.txt"
MAX_TERMINOS = 3000  # reducido para ahorrar RAM

def limpiar_palabra(palabra):
    palabra = ''.join(
        c for c in unicodedata.normalize('NFD', palabra)
        if unicodedata.category(c) != 'Mn'
    )
    palabra = palabra.lower()
    return ''.join(c for c in palabra if c.isalnum())

def procesar_corpus_gutenberg(archivo):
    with open(archivo, "r", encoding="utf-8", errors="ignore") as f:
        texto = f.read()
    documentos_raw = re.split(r'=+\s*TITLE:', texto)
    corpus = []
    for doc in documentos_raw:
        palabras = [limpiar_palabra(p) for p in doc.split()]
        palabras = [p for p in palabras if p]
        if palabras:
            corpus.append(palabras[:300])  # máx 300 palabras por libro
    return corpus

def construir_vocabulario(corpus, max_terminos=3000):
    frecuencia_global = Counter()
    for doc in corpus:
        frecuencia_global.update(doc)
    vocabulario = [t for t, _ in frecuencia_global.most_common(max_terminos)]
    return sorted(vocabulario)

def calcular_tfidf(corpus, vocabulario):
    N = len(corpus)
    vocab_index = {t: i for i, t in enumerate(vocabulario)}
    df = [0] * len(vocabulario)
    for doc in corpus:
        for termino in set(doc):
            if termino in vocab_index:
                df[vocab_index[termino]] += 1
    idf = [math.log10(N / d) if d > 0 else 0 for d in df]
    matriz_tfidf = []
    for doc_id, doc in enumerate(corpus):
        contador = Counter(doc)
        total = len(doc)
        fila = [0.0] * len(vocabulario)
        for termino, frecuencia in contador.items():
            if termino in vocab_index:
                idx = vocab_index[termino]
                fila[idx] = (frecuencia / total) * idf[idx]
        matriz_tfidf.append(fila)
        if (doc_id + 1) % 100 == 0:
            print(f"  Procesados {doc_id+1}/{N} documentos...")
    return matriz_tfidf, idf, vocab_index

# --- Ejecución ---
print("Procesando corpus...")
corpus = procesar_corpus_gutenberg(ARCHIVO)
print(f"Documentos cargados: {len(corpus)}")

print("\nConstruyendo vocabulario...")
vocabulario = construir_vocabulario(corpus, MAX_TERMINOS)
print(f"Términos: {len(vocabulario)}")

print("\nCalculando TF-IDF...")
matriz_tfidf, idf, vocab_index = calcular_tfidf(corpus, vocabulario)
print(f"\nTF-IDF calculado correctamente")
print(f"Dimensión de la matriz: {len(matriz_tfidf)} x {len(vocabulario)}")

Procesando corpus...
Documentos cargados: 1000

Construyendo vocabulario...
Términos: 3000

Calculando TF-IDF...
  Procesados 100/1000 documentos...
  Procesados 200/1000 documentos...
  Procesados 300/1000 documentos...
  Procesados 400/1000 documentos...
  Procesados 500/1000 documentos...
  Procesados 600/1000 documentos...
  Procesados 700/1000 documentos...
  Procesados 800/1000 documentos...
  Procesados 900/1000 documentos...
  Procesados 1000/1000 documentos...

TF-IDF calculado correctamente
Dimensión de la matriz: 1000 x 3000


### Paso 4: Programar una función `buscar()` para el corpus `Gutenberg 1000`

In [2]:
def buscar(consulta, corpus, matriz_tfidf, vocab_index, top_k=5):
    palabras = [limpiar_palabra(p) for p in consulta.split()]
    indices = [vocab_index[p] for p in palabras if p in vocab_index]
    if not indices:
        print("No hay términos válidos en la consulta.")
        return
    resultados = []
    for doc_id, vector in enumerate(matriz_tfidf):
        score = sum(vector[idx] for idx in indices)
        resultados.append((doc_id, score))
    resultados.sort(key=lambda x: x[1], reverse=True)
    print(f"\n{'='*40}")
    print(f"RESULTADOS: '{consulta}'")
    print(f"{'='*40}\n")
    encontrados = 0
    for doc_id, score in resultados:
        if score > 0:
            preview = " ".join(corpus[doc_id][:40])
            print(f"Documento : {doc_id + 1}")
            print(f"Score     : {round(score, 6)}")
            print(f"Preview   : {preview}")
            print("-" * 60)
            encontrados += 1
            if encontrados >= top_k:
                break

# --- Búsquedas de prueba ---
buscar("love war king", corpus, matriz_tfidf, vocab_index)
buscar("science machine electricity", corpus, matriz_tfidf, vocab_index)
buscar("ocean ship captain", corpus, matriz_tfidf, vocab_index)


RESULTADOS: 'love war king'

Documento : 11
Score     : 0.043959
Preview   : the complete works of william shakespeare gutenberg id 100 the project gutenberg ebook of the complete works of william shakespeare this ebook is for the use of anyone anywhere in the united states and most other parts of the world
------------------------------------------------------------
Documento : 384
Score     : 0.039563
Preview   : loves labours lost gutenberg id 1510 the project gutenberg ebook of loves labours lost this ebook is for the use of anyone anywhere in the united states and most other parts of the world at no cost and with almost
------------------------------------------------------------
Documento : 442
Score     : 0.036441
Preview   : on war gutenberg id 1946 the project gutenberg ebook of on war by carl von clausewitz this ebook is for the use of anyone anywhere at no cost and with almost no restrictions whatsoever you may copy it give it
-----------------------------------------------